In [1]:
## Perform residual kriging to calibrate predicted rasters using station observations
#  

# Workflow per month:
#   1. Get training station residuals for that month
#    a. read monthly raster as df
#    b. read filtered insitu training data for that month
#    c. merge on points where stations are available
#    d. get residuals in the df
#   2. Normal-score transform the residuals
#   3. Krige to 1km grid using seasonal variogram parameters
#   4. Back-transform kriged surface to mm space
#   5. Add to predicted raster → calibrated raster
#   6. Save calibrated raster

In [2]:
# LIBRARIES
import os
import gc
import pandas as pd
import numpy as np
import glob
import rasterio
from rasterio.transform import from_origin
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import rioxarray as rxr
import xarray as xr
import geostatspy.geostats as geostats
import geostatspy.GSLIB as GSLIB
from pyproj import Proj
from datetime import datetime
from scipy.spatial import cKDTree
from tqdm import tqdm

# CONSTANTS 
GSLIB_SENTINEL = 9.966e+20    
UTM_PROJ       = "+proj=utm +zone=45 +ellps=WGS84 +datum=WGS84 +units=m +no_defs"
NODATA_OUT     = -9999.0
GRID_RES_KM    = 1.0    
MIN_VALID_DAYS = 20         #Minimum valid days of observations for selecting station for residual kriging 
TRAIN_STATIONS  = r"..\Data\Processed\Split_stations\train_stations.csv"

START_YEAR, START_MONTH = 2001, 1
END_YEAR, END_MONTH = 2005, 12

seasons = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}

model_it    = {'spherical': 1, 'exponential': 2, 'gaussian': 3}
tmin, tmax  = -9999, 9999

# PATHS
pred_raster_path = r"..\Data\Processed\PredictedRasters\Final_rasters"
var_params_path = r"..\Data\Processed\Variogram_analysis\variogram_params_singleDomain.csv"
temp_utm_residual_dir = r"..\Data\Processed\Calibration_kriging\Temp_utm_residual_files"
temp_wgs_residual_dir = r"..\Data\Processed\Calibration_kriging\Temp_wgs_residual_files"
output_dir = r"..\Data\Processed\Calibration_kriging\Calibrated_rasters"
insitu_csv_path = r"..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv"

os.makedirs(temp_utm_residual_dir, exist_ok = True)
os.makedirs(temp_wgs_residual_dir, exist_ok = True)

os.makedirs(output_dir, exist_ok = True)

In [3]:
# PREPROCESSING 

# load variogram parameters for all seasons
var_params_all = pd.read_csv(var_params_path).set_index('season')

# prepare monthly insitu data for training stations
insitu_raw = pd.read_csv(insitu_csv_path)
insitu_raw["date"]  = pd.to_datetime(insitu_raw[["year", "month", "day"]])
insitu_raw          = insitu_raw.rename(columns={"lat": "y", "long": "x",
                                                  "rainfall_sum": "insitu_prec"})
insitu_raw          = insitu_raw[insitu_raw["insitu_prec"] >= 0]  
 
insitu_monthly = (
    insitu_raw
    .groupby(["index_no", "y", "x", "year", "month"])
    .agg(
        insitu_monthly_total=("insitu_prec", "sum"),
        valid_days          =("insitu_prec", "count")
    )
    .reset_index()
)

n_before = len(insitu_monthly)
insitu_monthly = insitu_monthly[insitu_monthly["valid_days"] >= MIN_VALID_DAYS].copy()
n_after  = len(insitu_monthly)
print(f"  Station-months before completeness filter: {n_before:,}")
print(f"  Station-months after  (>= {MIN_VALID_DAYS} valid days): {n_after:,}")
print(f"  Excluded: {n_before - n_after:,} ({100*(n_before-n_after)/n_before:.1f}%)")
 
# Filter to analysis period
insitu_monthly = insitu_monthly[
    (insitu_monthly["year"]  >= START_YEAR) &
    (insitu_monthly["year"]  <= END_YEAR)
].copy()
insitu_monthly[["x", "y"]] = insitu_monthly[["x", "y"]].round(5)
 
# Train / test split
train_ids = set(pd.read_csv(TRAIN_STATIONS)["index_no"])
insitu_train_monthly = insitu_monthly[insitu_monthly["index_no"].isin(train_ids)].copy()


  Station-months before completeness filter: 23,456
  Station-months after  (>= 20 valid days): 23,356
  Excluded: 100 (0.4%)


In [4]:
#HELPERS
def get_season(month):
    for s, months in seasons.items():
        if month in months:
            return s
    return None

def coords_to_utm_km(lons, lats):
    proj = Proj(UTM_PROJ)
    x_m, y_m = proj(lons, lats)
    return x_m / 1000.0, y_m / 1000.0

def mask_gslib_sentinel(arr, threshold=1e10):
    return np.where(np.abs(arr) > threshold, np.nan, arr.astype(float))

def safe_nscore_lookup(ns_pred, lookup_df):
    lut = lookup_df.sort_values('nscore_value').drop_duplicates('nscore_value')
    return np.interp(ns_pred, lut['nscore_value'].values, lut['original_value'].values)

In [5]:
# 1. Get training station residuals for that month
month_range = pd.period_range(
    start = f"{START_YEAR}-{START_MONTH:02d}",
    end   = f"{END_YEAR}-{END_MONTH:02d}",
    freq  = "M",
)

for p in tqdm(month_range, desc="Calibrate rasters"):
    # a. read monthly rasters as df
    year, month = p.year, p.month
    season = get_season(month)
    month_rxr = rxr.open_rasterio(os.path.join(pred_raster_path, f"Predicted_Raster_{year}_{month:02d}.tif")) 
    month_rxr.name = "pred_prec" 
    month_df = month_rxr.to_dataframe().reset_index()
    month_df = month_df.round(5)

    calibrated_path = os.path.join(output_dir, f"Calibrated_Raster_{year}_{month:02d}.tif")

    if os.path.exists(calibrated_path):
        print(f"skipped {calibrated_path} because file exists")
        continue

    x_km_full, y_km_full = coords_to_utm_km(month_df['x'], month_df['y'])
    xmn = float(x_km_full.min()) + GRID_RES_KM / 2
    ymn = float(y_km_full.min()) + GRID_RES_KM / 2
    nx  = int(round((x_km_full.max() - xmn) / GRID_RES_KM)) + 1
    ny  = int(round((y_km_full.max() - ymn) / GRID_RES_KM)) + 1

    # b. read filtered insitu training data for that month
    insitu_df = insitu_train_monthly[(insitu_train_monthly['year'] == year) & (insitu_train_monthly['month'] == month)]

    #c. merge on points where stations are available (which use cKD tree to search and match spatially closest pixels)
    tree = cKDTree(month_df[['x','y']].values)
    dist_tr, idx_tr = tree.query(insitu_df[['x', 'y']].values, k = 1)
    mask_tr = dist_tr <= 0.006
    utm_path = os.path.join(temp_utm_residual_dir, f"utm_{year}_{month}.tif")
    wgs_path = os.path.join(temp_wgs_residual_dir, f"wgs_{year}_{month}.tif")


    if mask_tr.any():
        station_rows = insitu_df.iloc[mask_tr].reset_index(drop=True)
        grid_rows = month_df.iloc[idx_tr[mask_tr]].reset_index(drop=True)

        merged_df = pd.concat([
            station_rows[['index_no', 'y', 'x', 'year', 'month', 'insitu_monthly_total']],
            grid_rows.drop(columns=['x','y', 'spatial_ref'])
        ], axis = 1) 
    else:
        print("no match found")

    merged_df = merged_df.drop(columns = ['band', 'year','month'])

    # d. get residuals in the df
    merged_df['residual'] = merged_df['insitu_monthly_total'] - merged_df['pred_prec']

    # 2. Normal-score transform the residuals
    ns_data, t_vals, t_ns = geostats.nscore(merged_df, 'residual')
    merged_df['ns'] = ns_data
    lookup_df = pd.DataFrame({'original_value': t_vals, 'nscore_value': t_ns})

    #load variogram parameters for the season
    vp = var_params_all.loc[season].copy()
    h_range = vp['range_km'] = vp['range_km']
    vario = GSLIB.make_variogram(
        nug = 0.15, nst=1, it1 = model_it[vp['model_type']],
        cc1 = vp['partial_sill'], azi1 = vp['azimuth_major'],
        hmaj1 = h_range, hmin1 = h_range
    )

    #get utm 45n x, y coordinates in merged_df to perform kriging
    merged_df['x_km'], merged_df['y_km'] = coords_to_utm_km(merged_df['x'], merged_df['y'])


    #   3. Krige to 1km grid using seasonal variogram parameters
    kmap, _ = geostats.kb2d(
        df = merged_df, xcol = 'x_km', ycol = 'y_km', vcol = 'ns',
        tmin = tmin, tmax = tmax, nx = nx, xmn = xmn, xsiz = GRID_RES_KM,
        ny = ny, ymn = ymn, ysiz = GRID_RES_KM, nxdis = 1, nydis = 1,
        ndmin = 1, ndmax = 12, radius = float(h_range), ktype = 1, skmean = 0, vario = vario 
    )

    kmap = mask_gslib_sentinel(kmap)

    if np.isnan(kmap).all():
        print("No nodes estimated")

    x_nodes = xmn + np.arange(nx) * GRID_RES_KM
    y_nodes = ymn + (ny - 1 - np.arange(ny)) * GRID_RES_KM
    xx, yy = np.meshgrid(x_nodes, y_nodes)

    #   4. Back-transform kriged surface to mm space

    out = pd.DataFrame({'x_km': xx.ravel(), 'y_km': yy.ravel(), 'ns_pred': kmap.ravel()}).dropna()
    out['residual_pred'] = safe_nscore_lookup(out['ns_pred'].values, lookup_df)

    all_x, all_y = np.sort(out['x_km'].unique()), np.sort(out['y_km'].unique())[::-1]
    grid_pivot = out.pivot(index= 'y_km', columns='x_km', values='residual_pred').reindex(index=all_y, columns=all_x)
    out_grid = np.where(np.isnan(grid_pivot.values), NODATA_OUT, grid_pivot.values).astype('float32')

    transform_utm = from_origin(
        west = all_x.min()*1000 - (GRID_RES_KM*500),
        north = all_y.max()*1000 + (GRID_RES_KM*500),
        xsize = GRID_RES_KM*1000, ysize=GRID_RES_KM*1000  
    )

    with rasterio.open(utm_path, 'w', driver='GTiff', height=out_grid.shape[0], width=out_grid.shape[1],
            count=1, dtype='float32', crs='EPSG:32645', transform=transform_utm, nodata=NODATA_OUT) as dst:
        dst.write(out_grid, 1)

    with rasterio.open(utm_path) as src:
        t_wgs, w, h = calculate_default_transform(src.crs, 'EPSG:4326', src.width, src.height, *src.bounds)
        out_wgs = np.full((1, h, w), NODATA_OUT, dtype='float32')
        reproject(
            source=rasterio.band(src, 1), destination=out_wgs, src_transform=src.transform,
            src_crs=src.crs, dst_transform=t_wgs, dst_crs='EPSG:4326',
            resampling=Resampling.bilinear, src_nodata=NODATA_OUT, dst_nodata=NODATA_OUT
        )
        profile = src.profile.copy()
        profile.update(crs='EPSG:4326', transform=t_wgs, width=w, height=h)
        if os.path.exists(wgs_path):
            os.remove(wgs_path)
        with rasterio.open(wgs_path, 'w', **profile) as dst:
            dst.write(out_wgs)

    if os.path.exists(utm_path):
        os.remove(utm_path)
        
    print(f"Kriged Residual Saved to {wgs_path}")

    #   5. Add to predicted raster
    residual_rxr = rxr.open_rasterio(wgs_path)
    residual_rxr = residual_rxr.rio.reproject_match(month_rxr, resampling = Resampling.bilinear)
    calibrated_rxr = month_rxr + residual_rxr
    calibrated_rxr = xr.where(calibrated_rxr < 0, 0, calibrated_rxr, keep_attrs= True)
    calibrated_rxr.rio.write_crs("EPSG:4326", inplace=True)
    calibrated_rxr.rio.to_raster(calibrated_path, encoded_nodata = -9999)

    if os.path.exists(wgs_path):
        gc.collect()
        os.remove(wgs_path)

Calibrate rasters:   3%|▎         | 2/60 [00:00<00:09,  6.06it/s]

skipped ..\Data\Processed\Calibration_kriging\Calibrated_rasters\Calibrated_Raster_2001_01.tif because file exists
skipped ..\Data\Processed\Calibration_kriging\Calibrated_rasters\Calibrated_Raster_2001_02.tif because file exists


Calibrate rasters:   7%|▋         | 4/60 [00:00<00:08,  6.54it/s]

skipped ..\Data\Processed\Calibration_kriging\Calibrated_rasters\Calibrated_Raster_2001_03.tif because file exists
skipped ..\Data\Processed\Calibration_kriging\Calibrated_rasters\Calibrated_Raster_2001_04.tif because file exists


Calibrate rasters:  10%|█         | 6/60 [00:00<00:08,  6.22it/s]

...

Calibrate rasters:  98%|█████████▊| 59/60 [44:28<01:34, 94.36s/it]

 make_variogram Warning: sill does not sum to 1.0, do not use in simulation 


100%|██████████| 513/513 [01:34<00:00,  5.41it/s]


  Estimated   452979 blocks 
      average   -0.06765982000013335  variance  0.3413282659445692
Kriged Residual Saved to ..\Data\Processed\Calibration_kriging\Temp_wgs_residual_files\wgs_2005_12.tif


Calibrate rasters: 100%|██████████| 60/60 [46:03<00:00, 46.06s/it]
